# RealVisXL V4.0 True Image-to-Image Studio
Fixed Image-to-Image pipeline so it actually uses your source image accurately.

In [ ]:
!pip install -q diffusers transformers accelerate torch torchvision gradio

import torch
from diffusers import AutoPipelineForImage2Image, AutoPipelineForText2Image
import gradio as gr

print("Loading RealVisXL V4.0 True Image-to-Image Pipeline...")
pipe_i2i = AutoPipelineForImage2Image.from_pretrained(
    "SG161222/RealVisXL_V4.0",
    torch_dtype=torch.float16,
    use_safetensors=True
)
pipe_i2i.enable_sequential_cpu_offload()

def edit(image, prompt, negative_prompt, strength, steps, cfg):
    if image is None:
        return None
    
    # Resize image to max 1024x1024 to fit SDXL perfectly
    image = image.convert("RGB")
    image.thumbnail((1024, 1024))
    
    full_prompt = prompt + ", raw photo, highly detailed, 8k uhd, photorealistic"
    full_neg = negative_prompt + ", blurry, low quality, deformed, ugly"
    
    print(f"Editing image with strength {strength}, steps {steps}, CFG {cfg}...")
    img = pipe_i2i(
        prompt=full_prompt,
        negative_prompt=full_neg,
        image=image,
        strength=float(strength),
        num_inference_steps=int(steps),
        guidance_scale=float(cfg)
    ).images[0]
    return img

demo = gr.Interface(
    fn=edit,
    inputs=[
        gr.Image(type="pil", label="Source Image (Your Photo)"),
        gr.Textbox(label="Prompt / What to change into", value="photorealistic, high quality portrait"),
        gr.Textbox(label="Negative Prompt", value="blurry, low quality"),
        gr.Slider(0.1, 0.9, value=0.45, label="Denoising Strength (0.2 = keep source structure, 0.8 = change completely)"),
        gr.Slider(20, 50, value=30, step=1, label="Steps"),
        gr.Slider(4.0, 10.0, value=7.5, step=0.5, label="CFG Scale")
    ],
    outputs=gr.Image(label="Edited Result"),
    title="RealVisXL V4.0 True Image-to-Image Studio"
)
demo.launch(share=True, debug=True)